# Baseline3 Adding Data Image Captions (model-reasoning based)

- after finishing the BOTH can dot this work.
- idea: 模型改进，这里的模型用的是很老的 20年 meta 的模型 "fairseq" => 拿它的生成器 "generator"

## Package Loading

In [ ]:
# ! pip install cython hydra-core omegaconf sacrebleu
# ! pip install git+https://github.com/pytorch/fairseq.git

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from fairseq import utils,tasks
from fairseq import checkpoint_utils
from utils.eval_utils import eval_step
from tasks.mm_tasks.caption import CaptionTask
from models.ofa import OFAModel
from PIL import Image

from tqdm.auto import tqdm

# Register caption task
tasks.register_task('caption',CaptionTask)

# turn on cuda if GPU is available
use_cuda = torch.cuda.is_available()
# use fp16 only when GPU is available
use_fp16 = False

- 检测计算资源

In [1]:
! pip install psutil torch

In [6]:
import psutil
import torch

def get_system_resources():
    # 获取CPU信息
    cpu_count = psutil.cpu_count(logical=False)  # 物理CPU核心数
    cpu_freq = psutil.cpu_freq()  # CPU频率
    memory = psutil.virtual_memory()  # 内存使用情况
    swap = psutil.swap_memory()  # 交换内存使用情况

    # 获取GPU信息（仅适用于安装了PyTorch的情况下）
    gpus = []
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            gpu_info = {}
            
            # 获取 GPU 名称
            gpu_info['GPU'] = torch.cuda.get_device_name(i)

            # 获取总内存
            try:
                gpu_info['Total Memory (MB)'] = torch.cuda.get_device_properties(i).total_memory / 1024**2
            except Exception as e:
                gpu_info['Total Memory (MB)'] = None
            
            # 获取分配的内存
            try:
                gpu_info['Memory Allocated (MB)'] = torch.cuda.memory_allocated(i) / 1024**2
            except Exception as e:
                gpu_info['Memory Allocated (MB)'] = None
            
            # 获取缓存的内存
            try:
                gpu_info['Memory Cached (MB)'] = torch.cuda.memory_reserved(i) / 1024**2
            except Exception as e:
                gpu_info['Memory Cached (MB)'] = None
            
            gpus.append(gpu_info)
    else:
        gpus.append({'GPU': 'No GPU available'})

    # 打印系统资源信息
    print("CPU Info:")
    print(f"  Physical cores: {cpu_count}")
    print(f"  Max Frequency: {cpu_freq.max} MHz")
    print(f"  Current Frequency: {cpu_freq.current} MHz")
    print(f"  Memory Total: {memory.total / 1024**3:.2f} GB")
    print(f"  Memory Available: {memory.available / 1024**3:.2f} GB")
    print(f"  Swap Total: {swap.total / 1024**3:.2f} GB")
    print(f"  Swap Used: {swap.used / 1024**3:.2f} GB")

    print("\nGPU Info:")
    for gpu in gpus:
        print(f"  GPU: {gpu.get('GPU', 'N/A')}")
        
        total_memory = gpu.get('Total Memory (MB)', 'N/A')
        print(f"    Total Memory: {total_memory if total_memory == 'N/A' else f'{total_memory:.2f} MB'}")
        
        memory_allocated = gpu.get('Memory Allocated (MB)', 'N/A')
        print(f"    Memory Allocated: {memory_allocated if memory_allocated == 'N/A' else f'{memory_allocated:.2f} MB'}")
        
        memory_cached = gpu.get('Memory Cached (MB)', 'N/A')
        print(f"    Memory Cached: {memory_cached if memory_cached == 'N/A' else f'{memory_cached:.2f} MB'}")

# 调用函数
get_system_resources()

CPU Info:
  Physical cores: 8
  Max Frequency: 3504 MHz
  Current Frequency: 3504 MHz
  Memory Total: 8.00 GB
  Memory Available: 1.38 GB
  Swap Total: 5.00 GB
  Swap Used: 4.30 GB

GPU Info:
  GPU: No GPU available
    Total Memory: N/A
    Memory Allocated: N/A
    Memory Cached: N/A


## Model Loading and Implement

- 超参数调配: 
> overrides={"bpe_dir":"utils/BPE", "eval_cider":False, "beam":5, "max_len_b":16, "no_repeat_ngram_size":3, "seed":7}
- 模型加载：
    - load => load model => ensemble task

- models (本方法应该融合了多种模型)
- 改LLM模型，这里的模型太老了...

In [ ]:
overrides={"bpe_dir":"utils/BPE", "eval_cider":False, "beam":5, "max_len_b":16, "no_repeat_ngram_size":3, "seed":7}
models, cfg, task = checkpoint_utils.load_model_ensemble_and_task(
        utils.split_paths('checkpoints/caption.pt'),
        arg_overrides=overrides
    )

# Move models to GPU
for model in models:
    model.eval()
    if use_fp16:
        model.half()
    if use_cuda and not cfg.distributed_training.pipeline_model_parallel:
        model.cuda()
    model.prepare_for_inference_(cfg)

# Initialize generator
generator = task.build_generator(models, cfg.generation)

## Preprocessing

- 图像处理
- txt prompt 并没有进行优化引导
    - idea: 可以进行 prompt 优化

In [ ]:
# Image transform
from torchvision import transforms
mean = [0.5, 0.5, 0.5]
std = [0.5, 0.5, 0.5]

patch_resize_transform = transforms.Compose([
    lambda image: image.convert("RGB"),
    transforms.Resize((cfg.task.patch_image_size, cfg.task.patch_image_size), interpolation=Image.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

# Text preprocess
bos_item = torch.LongTensor([task.src_dict.bos()])
eos_item = torch.LongTensor([task.src_dict.eos()])
pad_idx = task.src_dict.pad()
def encode_text(text, length=None, append_bos=False, append_eos=False):
    s = task.tgt_dict.encode_line(
        line=task.bpe.encode(text),
        add_if_not_exist=False,
        append_eos=False
    ).long()
    if length is not None:
        s = s[:length]
    if append_bos:
        s = torch.cat([bos_item, s])
    if append_eos:
        s = torch.cat([s, eos_item])
    return s

# Construct input for caption task
# 直接读图，处理图
def construct_sample(image: Image):
    patch_image = patch_resize_transform(image).unsqueeze(0)
    patch_mask = torch.tensor([True])
    src_text = encode_text(" what does the image describe?", append_bos=True, append_eos=True).unsqueeze(0)
    src_length = torch.LongTensor([s.ne(pad_idx).long().sum() for s in src_text])
    sample = {
        "id":np.array(['42']),
        "net_input": {
            "src_tokens": src_text,
            "src_lengths": src_length,
            "patch_images": patch_image,
            "patch_masks": patch_mask
        }
    }
    return sample
  
# Function to turn FP32 to FP16
def apply_half(t):
    if t.dtype is torch.float32:
        return t.to(dtype=torch.half)
    return t

## Run Reference and Reasoning

In [ ]:
info_fp = "../../data/hateful_memes/info_fine_grained.csv"
info_df = pd.read_csv(info_fp)
info_df.head()

## Question: 哪来的 hateful_memes_masked?
- 一个存储图像的文件夹：(但为啥有 masked?)

In [ ]:
# 路径修改
img_folder = "../../data/hateful_memes_masked"

captions = []
for img_fn in tqdm(info_df['img'].str.split('/').str[1]):
    img_fp = os.path.join(img_folder, img_fn)

    img = Image.open(img_fp)

    # Construct input sample & preprocess for GPU if cuda available
    sample = construct_sample(img)
    sample = utils.move_to_cuda(sample) if use_cuda else sample
    sample = utils.apply_to_sample(apply_half, sample) if use_fp16 else sample

    # Run eval step for caption
    with torch.no_grad():
        result, scores = eval_step(task, generator, models, sample)

    captions.append(result[0]['caption'])

    plt.imshow(img)
    plt.title(result[0]['caption'])
    plt.show()

In [ ]:
info_df['caption'] = captions
float_cols = info_df.select_dtypes(float).columns
info_df[float_cols] = info_df.select_dtypes(float).astype('Int64')

In [ ]:
info_df.to_csv("../../data/hateful_memes/hateful_memes_expanded.csv")